In [0]:
%sql
CREATE OR REPLACE FUNCTION pep_dev.gold.tenure_verdict(
    death_date DATE,
    end_date DATE,
    last_seen STRING,
    status_raw STRING,
    latest STRING
)

RETURNS STRING
RETURN CASE 
        WHEN death_date IS NOT NULL
            THEN 'confirmed_ended'
        WHEN end_date<date_format(to_date(left(last_seen,8),'yyyyMMdd'),'yyyy-MM-dd')
            THEN 'confirmed_ended'
        WHEN lower(status_raw)='ended'
            THEN 'confirmed_ended'
        WHEN (lower(status_raw)='current' OR end_date IS NOT NULL)
             AND last_seen=latest 
            THEN 'confirmed_current'
        WHEN last_seen<latest
            THEN 'derived_ended'
        ELSE 'unverifiable'
    END;
    

In [0]:
%sql
CREATE OR REPLACE FUNCTION pep_dev.gold.country_fetch(
    countries ARRAY<STRING>
)
RETURNS STRING
RETURN 
    CASE
        WHEN size(countries)=1 THEN countries[0]
        WHEN size(countries)=0 OR countries IS NULL THEN NULL    
        WHEN size(countries)>1 THEN 'multiple'
    END;

In [0]:
%sql

CREATE TABLE IF NOT EXISTS  pep_dev.gold.fact_occupancy_period(
    occupancy_id STRING NOT NULL,
    person_sk BIGINT NOT NULL,
    post_sk BIGINT NOT NULL,
    start_date_key INT,
    start_date_raw STRING,
    end_date_key INT,
    end_date_raw STRING,
    status STRING,
    tenure_verdict STRING NOT NULL,
    source_count INT NOT NULL,
    _first_seen_snapshot STRING NOT NULL,
    _last_seen_snapshot STRING NOT NULL,
    CONSTRAINT fact_occupancy_period_pk PRIMARY KEY (occupancy_id)

) USING DELTA ;

ALTER TABLE pep_dev.gold.fact_occupancy_period
    DROP CONSTRAINT IF EXISTS tenure_verdict_valid;

ALTER TABLE pep_dev.gold.fact_occupancy_period
    ADD CONSTRAINT tenure_verdict_valid
    CHECK(tenure_verdict IN ('confirmed_current', 'confirmed_ended', 'derived_ended', 'unverifiable'));

In [0]:
%sql

CREATE TABLE IF NOT EXISTS pep_dev.gold.dim_person
(
    person_sk BIGINT NOT NULL GENERATED BY DEFAULT AS IDENTITY,
    person_id STRING NOT NULL,
    primary_name STRING NOT NULL,
    birth_year_min INT,
    birth_year_max INT,
    death_date DATE,
    is_pep boolean NOT NULL,
    is_sanctioned boolean NOT NULL,
    is_current_pep boolean NOT NULL,
    n_positions INT NOT NULL,
    CONSTRAINT dim_person_pk PRIMARY KEY (person_sk)

) USING DELTA;


In [0]:
%sql
CREATE TABLE IF NOT EXISTS  pep_dev.gold.dim_post
(
    post_sk BIGINT NOT NULL GENERATED BY DEFAULT AS IDENTITY,
    post_id STRING NOT NULL,
    post_name STRING NOT NULL,
    country STRING,
    CONSTRAINT dim_post_pk PRIMARY KEY (post_sk)

) USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS pep_dev.gold.dim_date (
    date_key    INT     NOT NULL,                       -- CHANGED: added NOT NULL
    full_date   DATE    NOT NULL,
    year        INT     NOT NULL,
    quarter     INT     NOT NULL,
    month       INT     NOT NULL,
    month_name  STRING  NOT NULL,
    CONSTRAINT dim_date_pk PRIMARY KEY (date_key)       -- ADDED
) USING DELTA;                                          -- ADDED

In [0]:
%sql
CREATE TABLE IF NOT EXISTS pep_dev.gold.ml_tenure_features (
    occupancy_id          STRING   NOT NULL,
    person_id             STRING   NOT NULL,
    _snapshot_version     STRING   NOT NULL,
    label_is_current      BOOLEAN,
    has_start_date        BOOLEAN  NOT NULL,
    start_year            INT,
    years_since_start     INT,
    source_count          INT      NOT NULL,
    post_country          STRING,
    person_birth_year     INT,
    person_n_positions    INT      NOT NULL,
    person_is_sanctioned  BOOLEAN  NOT NULL,
    CONSTRAINT ml_tenure_features_pk PRIMARY KEY (occupancy_id)
) USING DELTA;